# nano-deepseek-v4 — Tutorial 01
## Loading and Inspecting the Official Flash Checkpoint

This notebook walks through:

1. Building the tiny CPU-runnable default model in a few lines.
2. Downloading the official DeepSeek-V4-Flash snapshot from Hugging Face.
3. Running snapshot preflight (shards, keys, shapes, dtype metadata).
4. Building the Flash architecture from the official `config.json`.
5. Loading and converting the official safetensors shards into our model layout.
6. Sanity-checking a few weights and dimensions.

Total runtime (excluding the ~149 GB download): under 10 minutes on a single A100/H100/4090.
CPU-only sections run in under a minute.

## 0. Imports and sanity

In [ ]:
import torch

import nano_deepseek_v4 as ndv4

print('nano_deepseek_v4 version:', ndv4.__version__)
print('torch version:           ', torch.__version__)
print('cuda available:          ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device:             ', torch.cuda.get_device_name(0))

## 1. The tiny default model

`DeepSeekV4Config()` (no args) gives a ~1M-parameter toy that runs on CPU.
It still uses **every** architectural component of the full model — hybrid attention,
Manifold-Constrained Hyper-Connections, hash-MoE + routed MoE, MTP head — just at
tiny dimensions.

This is what `pytest tests/` exercises.

In [ ]:
torch.manual_seed(0)

tiny_cfg = ndv4.DeepSeekV4Config()
tiny_model = ndv4.DeepSeekV4ForCausalLM(tiny_cfg)

n_params = sum(p.numel() for p in tiny_model.parameters())
print(f'parameters:       {n_params:,}')
print(f'vocab_size:       {tiny_cfg.vocab_size}')
print(f'hidden_size:      {tiny_cfg.hidden_size}')
print(f'num_layers:       {tiny_cfg.num_hidden_layers}')
print(f'n_routed_experts: {tiny_cfg.n_routed_experts}')
print(f'layer_types:      {tiny_cfg.layer_types}')
print(f'mlp_layer_types:  {tiny_cfg.mlp_layer_types}')

In [ ]:
# Forward pass with random input
ids = torch.randint(0, tiny_cfg.vocab_size, (1, 16))
out = tiny_model(ids, labels=ids)

print(f'logits shape:  {tuple(out.logits.shape)}')
print(f'training loss: {out.loss.item():.4f}')
print(f'mtp loss:      {out.mtp_loss.item():.4f}' if out.mtp_loss is not None else 'mtp loss: None')

## 2. The official Flash and Pro configs

`DeepSeekV4Config.flash()` and `DeepSeekV4Config.pro()` reproduce the published
architecture exactly. We can inspect them without downloading any weights.

In [ ]:
flash_cfg = ndv4.DeepSeekV4Config.flash()
pro_cfg = ndv4.DeepSeekV4Config.pro()

for name, cfg in [('Flash', flash_cfg), ('Pro', pro_cfg)]:
    print(f'--- {name} ---')
    print(f'  hidden_size:           {cfg.hidden_size}')
    print(f'  num_hidden_layers:     {cfg.num_hidden_layers}')
    print(f'  num_attention_heads:   {cfg.num_attention_heads}')
    print(f'  head_dim:              {cfg.head_dim}')
    print(f'  n_routed_experts:      {cfg.n_routed_experts}')
    print(f'  num_experts_per_tok:   {cfg.num_experts_per_tok}')
    print(f'  max_position_embed:    {cfg.max_position_embeddings:,}')
    layer_types_summary = {}
    for t in cfg.layer_types:
        layer_types_summary[t] = layer_types_summary.get(t, 0) + 1
    print(f'  layer_types:           {layer_types_summary}')

In [ ]:
# Parameter count estimate (without instantiating the multi-hundred-billion-parameter model)
flash_counts = ndv4.estimate_deepseek_v4_parameter_counts(flash_cfg)
pro_counts = ndv4.estimate_deepseek_v4_parameter_counts(pro_cfg)

for name, counts in [('Flash', flash_counts), ('Pro', pro_counts)]:
    total_b = counts['total_parameters'] / 1e9
    active_b = counts['activated_parameters'] / 1e9
    print(f'{name}:  total={total_b:>7.1f}B   activated/token={active_b:>5.1f}B')

## 3. Downloading the Flash snapshot

Flash is **~160 GB (149 GiB)** across 46 safetensors shards (FP4/FP8 quantized).
If you already have the snapshot, point `FLASH_DIR` at it.
Otherwise, the cell below will use `hf download`.

Pro is **~865 GB** across 64 shards. Both models are open-weight; see
[deepseek-ai/DeepSeek-V4-Flash](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash)
and [deepseek-ai/DeepSeek-V4-Pro](https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro).

In [ ]:
from pathlib import Path

FLASH_DIR = Path('./checkpoints/flash')  # adjust if you have it elsewhere

if not (FLASH_DIR / 'config.json').exists():
    print(f'Flash snapshot not found at {FLASH_DIR}.')
    print('To download (~160 GB / 149 GiB, requires sufficient disk):')
    print('  hf download deepseek-ai/DeepSeek-V4-Flash \\')
    print(f'      --local-dir {FLASH_DIR}')
    print('Skip the rest of this notebook until the download is complete.')
else:
    print(f'Flash snapshot found at {FLASH_DIR}.')
    files = sorted(FLASH_DIR.glob('*'))
    print(f'  files: {len(files)}')
    shards = sorted(FLASH_DIR.glob('model-*.safetensors'))
    print(f'  safetensors shards: {len(shards)}')

## 4. Snapshot preflight

`verify_deepseek_checkpoint_snapshot` checks:
- all shards present,
- `model.safetensors.index.json` consistent,
- expected config-derived keys present in the index,
- safetensors metadata declares the expected tensor shapes and dtypes,
- U8 quantized tensors have their FP4/FP8 scale sidecars.

It does **not** read tensor payloads — fast, cheap, runs in seconds.

In [ ]:
if (FLASH_DIR / 'config.json').exists():
    report = ndv4.verify_deepseek_checkpoint_snapshot(str(FLASH_DIR))
    print(f'is_complete:         {report.is_complete}')
    print(f'total_keys:          {report.total_keys}')
    print(f'total_shards:        {report.total_shards}')
    print(f'total_tensor_bytes:  {report.total_tensor_bytes:,}')
    print(f'missing_shards:      {report.missing_shards}')
    print(f'shape_mismatches:    {report.shape_mismatches}')

## 5. Index coverage by tensor name pattern

We can also inspect the official index by tensor-name pattern, to understand which
groups of weights are present (embed, attention LoRA, mHC, MoE experts, MTP, etc.).

In [ ]:
if (FLASH_DIR / 'model.safetensors.index.json').exists():
    coverage = ndv4.analyze_deepseek_official_index(
        FLASH_DIR / 'model.safetensors.index.json'
    )
    print(f'total keys in index:  {coverage.total_keys}')
    print(f'unique patterns:      {len(coverage.pattern_counts)}')
    print()
    print('top 20 pattern groups:')
    for pattern, count in sorted(
        coverage.pattern_counts.items(),
        key=lambda kv: -kv[1],
    )[:20]:
        print(f'  {count:>5d}  {pattern}')

## 6. Building Flash architecture from official config

`DeepSeekV4Config.from_official_json` reads the snapshot's `config.json` and
builds a config object whose layer schedule, dimensions, and routing settings
match the published model exactly.

Instantiating the full 284B Flash model **requires GPU memory** (or sharded loading).
If you have an 80GB+ GPU, you can do this directly. On a smaller GPU you'll need to
use staged loading; for inspection-only purposes you can also build the model on CPU
with bf16.

In [ ]:
if (FLASH_DIR / 'config.json').exists():
    flash_cfg_from_snapshot = ndv4.DeepSeekV4Config.from_official_json(
        FLASH_DIR / 'config.json'
    )
    print('Loaded config from snapshot:')
    print(f'  hidden_size:       {flash_cfg_from_snapshot.hidden_size}')
    print(f'  num_hidden_layers: {flash_cfg_from_snapshot.num_hidden_layers}')
    print(f'  n_routed_experts:  {flash_cfg_from_snapshot.n_routed_experts}')
    # Confirm it matches the built-in flash() preset
    assert flash_cfg_from_snapshot.hidden_size == flash_cfg.hidden_size
    assert flash_cfg_from_snapshot.num_hidden_layers == flash_cfg.num_hidden_layers
    print('\n✓ matches built-in DeepSeekV4Config.flash() preset')

## 7. Loading and converting the official shards

`load_deepseek_official_checkpoint` performs the full pipeline:

- load all 46 safetensors shards (lazy, layer-by-layer),
- convert official tensor names into our model's expected key layout,
- dequantize U8 FP4 E2M1 / FP8 E4M3FN tensors using their scale sidecars,
- copy into the target model's `state_dict` with shape verification.

**Note:** instantiating the full Flash model in fp32 takes ~600GB RAM. Use
`torch.set_default_dtype(torch.bfloat16)` to halve that, or use a 80GB+ GPU.
The cell below illustrates the call but does not actually run it unless you
set `RUN_HEAVY_LOAD = True`.

In [ ]:
RUN_HEAVY_LOAD = False   # set True if you have the memory

if RUN_HEAVY_LOAD and (FLASH_DIR / 'config.json').exists():
    torch.set_default_dtype(torch.bfloat16)
    flash_model = ndv4.DeepSeekV4ForCausalLM(flash_cfg_from_snapshot)
    flash_model, conversion = ndv4.load_deepseek_official_checkpoint(
        flash_model, FLASH_DIR
    )
    print(f'converted_keys:   {conversion.converted_key_count}')
    print(f'unconverted_keys: {len(conversion.unconverted_keys)}')
    print(f'ignored_keys:     {len(conversion.ignored_keys)}')
    print(f'missing_keys:     {len(conversion.missing_keys)}')
    print(f'unexpected_keys:  {len(conversion.unexpected_keys)}')
else:
    print('Heavy load skipped. Flip RUN_HEAVY_LOAD to True with enough memory.')
    print('Expected on Flash: ~35,000 converted, ~34,000 ignored (scale sidecars).')

## 8. Where to go from here

- `02_architecture_tour.ipynb` — walk through the modeling.py components
  (CSA/HCA compression, mHC Sinkhorn projection, hash + routed MoE, MTP).
- `03_muon_optimizer.ipynb` — Muon optimizer vs AdamW on a small training run.
- Source code: `nano_deepseek_v4/` is ~4,300 lines. Start with `modeling.py`.
- DeepSeek-V4 paper: https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro/blob/main/DeepSeek_V4.pdf
- HF Transformers reference: https://huggingface.co/docs/transformers/main/model_doc/deepseek_v4

Issues or contributions welcome on the GitHub repo.